In [1]:
import os
import re
import PyPDF2
import openai
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from tqdm import tqdm
import concurrent.futures

# Configuration
PDF_FOLDER = "cot_papers_selected"
EMBEDDING_MODEL = "text-embedding-ada-002"
CHAT_MODEL = "gpt-4o-mini"
NUM_CLUSTERS = 10
OUTPUT_CSV = "taxonomy_assignments.csv"
MAX_WORKERS = min(32, (os.cpu_count() or 1) * 5)

# Load OpenAI API key from environment
openai.api_key = os.environ.get("OPENAI_API_KEY")

# Function to extract abstract (or title fallback) from a single PDF
def extract_abstract_title(fn):
    path = os.path.join(PDF_FOLDER, fn)
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        full_text = ""
        for page in reader.pages:
            try:
                full_text += page.extract_text() + "\n"
            except:
                continue
    full_text = re.sub(r'\s+', ' ', full_text).strip()
    lower_text = full_text.lower()
    abs_text = ""
    abstract_match = re.search(r'(?i)\babstract\b', lower_text)
    if abstract_match:
        start_idx = abstract_match.end()
        candidate = full_text[start_idx:]
        intro_match = re.search(r'(?i)\bintroduction\b', candidate)
        end_idx = intro_match.start() if intro_match else None
        snippet = candidate[:end_idx] if end_idx else candidate[:2000]
        abs_text = snippet.strip()
    if abs_text:
        title = next((line.strip() for line in full_text.split('.') if line.strip()), fn.replace(".pdf", ""))
        return abs_text.replace("\n", " "), title
    else:
        title = next((line.strip() for line in full_text.split('.') if line.strip()), fn.replace(".pdf", ""))
        return "", title

# Function to get embedding for a piece of text
def get_embedding(text):
    response = openai.Embedding.create(model=EMBEDDING_MODEL, input=text)
    return response["data"][0]["embedding"]

# Function to label a single cluster via LLM
def label_cluster(cluster_id):
    cluster_abstracts = cluster_data[cluster_id]
    prompt = (
        "You are an expert in AI research with deep knowledge of reasoning-model techniques. "
        "Below are the abstracts (or title as fallback) of several papers, all focused on reasoning methods. "
        "Create a fine-grained taxonomy category name for this group and list up to three subcategories "
        "that capture the nuanced themes within these methods. Respond in this format:\n\n"
        "Category: <category name>\n"
        "Subcategories:\n"
        "1. <subcat1>\n"
        "2. <subcat2>\n"
        "3. <subcat3>\n\n"
        "Here are the abstracts:\n\n"
        + "\n\n---\n\n".join(f"{i+1}. {cluster_abstracts[i]}" for i in range(len(cluster_abstracts)))
    )
    resp = openai.ChatCompletion.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=500
    )
    return cluster_id, resp["choices"][0]["message"]["content"].strip()

# 1. Read filenames
filenames = [f for f in os.listdir(PDF_FOLDER) if f.lower().endswith(".pdf")]

# 2. Extract abstracts and titles in parallel
abstracts = [None] * len(filenames)
titles = [None] * len(filenames)
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(extract_abstract_title, fn): idx for idx, fn in enumerate(filenames)}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(filenames), desc="Extracting abstracts"):
        idx = futures[future]
        abs_text, title = future.result()
        abstracts[idx] = abs_text
        titles[idx] = title

# 3. Generate embeddings in parallel (use abstract or title fallback)
texts_for_embedding = [abstracts[i] if abstracts[i] else titles[i] for i in range(len(filenames))]
embeddings = [None] * len(texts_for_embedding)
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(get_embedding, text): idx for idx, text in enumerate(texts_for_embedding)}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(texts_for_embedding), desc="Generating embeddings"):
        idx = futures[future]
        embeddings[idx] = future.result()
X = np.array(embeddings)

# 4. Cluster embeddings
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42).fit(X)
labels = kmeans.labels_

# 5. Prepare cluster data for labeling
cluster_data = {}
for cluster_id in range(NUM_CLUSTERS):
    idxs = [i for i, lbl in enumerate(labels) if lbl == cluster_id]
    cluster_data[cluster_id] = [abstracts[i] if abstracts[i] else titles[i] for i in idxs]

# 6. Label clusters in parallel
cluster_labels = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_CLUSTERS) as executor:
    futures = {executor.submit(label_cluster, cid): cid for cid in range(NUM_CLUSTERS)}
    for future in tqdm(concurrent.futures.as_completed(futures), total=NUM_CLUSTERS, desc="Labeling clusters"):
        cid, label_text = future.result()
        cluster_labels[cid] = label_text

# 7. Build DataFrame and save to CSV
rows = []
for fn, cluster_id in zip(filenames, labels):
    rows.append({
        "filename": fn,
        "cluster_id": cluster_id,
        "taxonomy_label": cluster_labels[cluster_id]
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved taxonomy assignments to {OUTPUT_CSV}")


Generating embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 179/179 [00:02<00:00, 60.65it/s]
/home/oppenheimer/anaconda3/envs/advisormatch/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
Labeling clusters: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:02<00:00,  4.38it/s]

Saved taxonomy assignments to taxonomy_assignments.csv


In [5]:
for val in list(cluster_labels.values()):
    print(val)

Category: Advanced Reasoning Techniques in AI

Subcategories:
1. Inference Optimization Methods
2. Process Supervised Reward Models
3. Multi-Hop Reasoning Frameworks
Category: Long Chain-of-Thought Reasoning Methods  
Subcategories:  
1. Action Space Constraining and Search Strategies  
2. Negative Sample Utilization and Policy Optimization  
3. Context Management and Memory Efficiency
Category: Reasoning Optimization Techniques
Subcategories:
1. Inference Efficiency Strategies
2. Adaptive Decomposition Methods
3. Self-Improvement and Distillation Approaches
Category: Efficient Reasoning Methods in Large Language Models  
Subcategories:  
1. Token Sparsity and Selection Techniques  
2. Adaptive Attention Mechanisms  
3. Cache Management and Context Preservation Strategies
Category: Reinforcement Learning Techniques for Reasoning in Language Models

Subcategories:
1. Reward Optimization Strategies
2. Efficiency and Computational Cost Reduction
3. Intermediate Reasoning Process Regulatio